# Dataset Cleaning: Split Multi-Part Questions & Normalize Answers

## Cell 1 — Setup

In [1]:
import json
import os
import re
import sys
import time
import threading
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed

from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

sys.path.insert(0, '..')
from utils.api_client import create_openrouter_client
from utils.data_io import load_dataset, save_dataset

# Load environment and create client with PROF key
load_dotenv()
api_key = os.getenv('OPENROUTER_API_KEY_PROF')
client = create_openrouter_client(api_key=api_key)
client = OpenAI(
    base_url=client.base_url,
    api_key=client.api_key,
    timeout=180,
)


@dataclass
class CleaningConfig:
    """Configuration for the cleaning pipeline."""
    model: str = 'anthropic/claude-opus-4.6'
    temperature: float = 0.1
    max_tokens: int = 8000
    max_workers: int = 5
    master_file: str = '../data/master_dataset.jsonl'
    output_file: str = '../data/master_dataset_cleaned.jsonl'
    checkpoint_file: str = '../data/cleaning_checkpoint.jsonl'


config = CleaningConfig()

# Thread-safe token counter
token_counts: Dict[str, int] = {}
token_lock = threading.Lock()


def track_tokens(model: str, usage) -> None:
    """Thread-safe token tracking per model."""
    if usage is None:
        return
    with token_lock:
        if model not in token_counts:
            token_counts[model] = 0
        token_counts[model] += (usage.prompt_tokens or 0) + (usage.completion_tokens or 0)


def extract_content_from_message(message) -> Optional[str]:
    """Extract usable text content from a chat completion message."""
    if message.content is not None:
        return message.content
    if getattr(message, 'refusal', None) is not None:
        print(f'  [!] model refused: {message.refusal}', flush=True)
        return None
    extras = getattr(message, 'model_extra', None) or {}
    for field in ('reasoning', 'reasoning_content', 'reasoning_details'):
        value = extras.get(field)
        if value and isinstance(value, str) and len(value.strip()) > 0:
            return value.strip()
        if value and isinstance(value, list):
            texts = [v.get('content', '') for v in value if isinstance(v, dict)]
            joined = '\n'.join(t for t in texts if t)
            if joined.strip():
                return joined.strip()
    return None


def call_model(prompt: str, config: CleaningConfig) -> str:
    """Call OpenRouter model with retry logic."""
    max_retries = 3
    retry_delay = 2.0

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=config.model,
                max_tokens=config.max_tokens,
                temperature=config.temperature,
                timeout=180,
                messages=[{'role': 'user', 'content': prompt}],
            )
            track_tokens(config.model, response.usage)
            content = extract_content_from_message(response.choices[0].message)
            if content is not None and content.strip():
                return content
            wait_time = retry_delay * (2 ** attempt)
            if attempt < max_retries - 1:
                print(f'  content empty (attempt {attempt+1}) -- retrying in {wait_time:.0f}s', flush=True)
                time.sleep(wait_time)
            else:
                return ''
        except Exception as e:
            wait_time = retry_delay * (2 ** attempt)
            if attempt < max_retries - 1:
                print(f'  attempt {attempt+1} failed: {e} -- retrying in {wait_time:.0f}s', flush=True)
                time.sleep(wait_time)
            else:
                print(f'  FAILED after {max_retries} attempts: {e}', flush=True)
                return ''
    return ''


print('Setup complete.')
print(f'  Model: {config.model}')
print(f'  Max workers: {config.max_workers}')

Setup complete.
  Model: anthropic/claude-opus-4.6
  Max workers: 5


## Cell 2 — Load Dataset

In [2]:
master_dataset = load_dataset(config.master_file)
print(f'Loaded {len(master_dataset)} items from {config.master_file}')

# Show a sample
if master_dataset:
    s = master_dataset[0]
    print(f'\nSample entry:')
    print(f'  Q: {s["question"][:120]}...')
    print(f'  A: {s["answer"][:80]}')
    print(f'  Source: {s.get("source", "N/A")}')

Loaded 140 items from ../data/master_dataset.jsonl

Sample entry:
  Q: What is the minimum sample size that will allow us to verify a 500,000-hour MTTF with 85% confidence, given that the tes...
  A: 380.
  Source: seed


## Cell 3 — LLM Prompt Design

In [3]:
CLEANING_PROMPT = """You are a data-quality assistant for a reliability-engineering Q&A dataset.

I will give you ONE dataset entry with fields: question, reasoning, answer.

Your job:

1. **Classify** the question as one of:
   - `MULTI_PART_NUMERIC` — asks for multiple numeric values (e.g. comma-separated answer)
   - `SINGLE_NUMERIC` — asks for exactly one numeric value
   - `FORMULA` — asks to derive or find a formula/expression
   - `DERIVATION` — asks to "show that" or prove something (the answer IS a formula)
   - `CONCEPTUAL` — asks for an explanation or interpretation

2. **Split** multi-part questions into N fully self-contained single-answer entries.
   For single-part questions, return 1 entry.

3. **Clean** every answer to be a pure value:
   - Strip trailing periods, units (like "hours", "units"), commas in numbers, HTML tags (`<sup>`, `<sub>`, etc.), and asterisk formatting (`*bold*`).
   - Convert percentage answers to decimal if they contain `%` (e.g. "67.03%" -> "0.6703").
   - For formulas: use clean LaTeX `$...$` notation. Reconstruct garbled formula answers using the reasoning field (which has clean LaTeX).
   - For conceptual/text: short phrase, no trailing punctuation.
   - Numeric answers should be plain numbers: "380", "0.95", "1000", "5.2e-6".

**Rules for splitting:**
- Each split question MUST include ALL given data, distributions, parameters — fully standalone. A reader must be able to solve it without seeing the other parts.
- Do NOT use phrases like "from the previous problem", "as above", "Part (a)", etc.
- Trim the reasoning to only the relevant sub-part for each split entry.
- For derivations ("show that X"), convert to a formula-type entry where the answer is the derived formula.
- For mixed conceptual+numeric questions, split them into separate entries.

**Return valid JSON** (no markdown fences, no extra text before/after):
```
{{
  "classification": "MULTI_PART_NUMERIC",
  "entries": [
    {{
      "question": "...",
      "reasoning": "...",
      "answer": "380",
      "answer_type": "numeric"
    }}
  ],
  "notes": "any concerns or empty string"
}}
```

`answer_type` must be one of: `numeric`, `formula`, `text`, `boolean`.

---

**ENTRY TO PROCESS:**

QUESTION:
{question}

REASONING:
{reasoning}

ANSWER:
{answer}
"""


def build_prompt(item: Dict) -> str:
    return CLEANING_PROMPT.format(
        question=item['question'],
        reasoning=item['reasoning'],
        answer=item['answer'],
    )


print('Prompt template defined.')
print(f'Prompt length (sample): ~{len(build_prompt(master_dataset[0]))} chars')

Prompt template defined.
Prompt length (sample): ~5001 chars


## Cell 4 — Processing Functions

In [4]:
def parse_json_response(text: str) -> Optional[Dict]:
    """Parse JSON from LLM response with fallbacks."""
    # Direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strip markdown code fences
    stripped = re.sub(r'^```(?:json)?\s*', '', text.strip())
    stripped = re.sub(r'\s*```$', '', stripped)
    try:
        return json.loads(stripped)
    except json.JSONDecodeError:
        pass

    # Regex: extract outermost {...}
    match = re.search(r'\{[\s\S]*\}', text)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    return None


def clean_answer_post(answer: str, answer_type: str) -> str:
    """Deterministic Python cleanup after LLM processing."""
    # Remove HTML tags
    answer = re.sub(r'<[^>]+>', '', answer)
    # Remove asterisk formatting
    answer = re.sub(r'\*+', '', answer)
    # Strip whitespace
    answer = answer.strip()
    # Remove trailing period(s)
    answer = answer.rstrip('.')
    answer = answer.strip()

    if answer_type == 'numeric':
        # Remove unit suffixes (common ones)
        answer = re.sub(r'\s*(hours?|units?|failures?|days?|years?|FITs?)\s*$', '', answer, flags=re.IGNORECASE)
        answer = answer.strip()
        # Remove commas in numbers: 5,323 -> 5323
        answer = re.sub(r'(\d),(\d{3})', r'\1\2', answer)
        # Repeat for larger numbers like 1,000,000
        answer = re.sub(r'(\d),(\d{3})', r'\1\2', answer)
        # Remove trailing period that may remain
        answer = answer.rstrip('.')

    if answer_type == 'text':
        # Remove trailing punctuation for text answers
        answer = answer.rstrip('.!;:')
        answer = answer.strip()

    return answer


def process_item(item: Dict, index: int, config: CleaningConfig) -> Dict:
    """Process a single dataset item: call LLM, parse response, clean answers."""
    prompt = build_prompt(item)
    raw_response = call_model(prompt, config)

    if not raw_response:
        return {
            'original_index': index,
            'status': 'error',
            'error': 'Empty LLM response',
            'entries': [],
        }

    parsed = parse_json_response(raw_response)
    if parsed is None:
        return {
            'original_index': index,
            'status': 'error',
            'error': 'JSON parse failed',
            'raw_response': raw_response[:500],
            'entries': [],
        }

    classification = parsed.get('classification', 'UNKNOWN')
    notes = parsed.get('notes', '')
    raw_entries = parsed.get('entries', [])

    cleaned_entries = []
    for entry in raw_entries:
        answer_type = entry.get('answer_type', 'text')
        cleaned_answer = clean_answer_post(entry.get('answer', ''), answer_type)
        cleaned_entries.append({
            'question': entry.get('question', ''),
            'reasoning': entry.get('reasoning', ''),
            'answer': cleaned_answer,
            'answer_type': answer_type,
            'source': item.get('source', 'unknown'),
            'original_index': index,
        })

    return {
        'original_index': index,
        'status': 'ok',
        'classification': classification,
        'notes': notes,
        'entries': cleaned_entries,
        'was_split': len(cleaned_entries) > 1,
    }


print('Processing functions defined.')

Processing functions defined.


## Cell 5 — Run Processing

In [5]:
# Load checkpoint if it exists (for resumability)
checkpoint_path = Path(config.checkpoint_file)
completed_indices = set()
checkpoint_results = []

if checkpoint_path.exists():
    for line in checkpoint_path.read_text(encoding='utf-8').strip().splitlines():
        if line.strip():
            result = json.loads(line)
            completed_indices.add(result['original_index'])
            checkpoint_results.append(result)
    print(f'Resuming: {len(completed_indices)} items already processed')

# Build work list (skip already-processed items)
work_items = [
    (i, item) for i, item in enumerate(master_dataset)
    if i not in completed_indices
]
print(f'Items to process: {len(work_items)} (of {len(master_dataset)} total)')

# Process with thread pool
results = list(checkpoint_results)  # start with checkpoint
stats = {'processed': len(completed_indices), 'split': 0, 'failed': 0}

checkpoint_lock = threading.Lock()


def process_and_save(args):
    idx, item = args
    result = process_item(item, idx, config)
    # Append to checkpoint file (thread-safe via file append)
    with checkpoint_lock:
        with open(config.checkpoint_file, 'a', encoding='utf-8') as f:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    return result


if work_items:
    with ThreadPoolExecutor(max_workers=config.max_workers) as executor:
        futures = {
            executor.submit(process_and_save, (idx, item)): idx
            for idx, item in work_items
        }
        with tqdm(total=len(work_items), desc='Cleaning') as pbar:
            for future in as_completed(futures):
                try:
                    result = future.result()
                    results.append(result)
                    stats['processed'] += 1
                    if result.get('was_split'):
                        stats['split'] += 1
                    if result.get('status') == 'error':
                        stats['failed'] += 1
                except Exception as e:
                    stats['failed'] += 1
                    print(f'  Exception: {e}', flush=True)
                pbar.update(1)

print(f'\nDone. Processed: {stats["processed"]}, Split: {stats["split"]}, Failed: {stats["failed"]}')

Resuming: 9 items already processed
Items to process: 131 (of 140 total)


Cleaning: 100%|██████████| 131/131 [05:09<00:00,  2.36s/it]


Done. Processed: 140, Split: 107, Failed: 0


## Cell 6 — Quality Checks

In [6]:
# Collect all cleaned entries
all_entries: List[Dict] = []
for result in results:
    if result.get('status') == 'ok':
        all_entries.extend(result['entries'])

print(f'Total cleaned entries: {len(all_entries)}')
print(f'Original items: {len(master_dataset)}')

# --- Answer format regex check ---
NUMERIC_RE = re.compile(r'^-?\d+\.?\d*(?:[eE][-+]?\d+)?$')
format_failures = []
for entry in all_entries:
    if entry['answer_type'] == 'numeric':
        if not NUMERIC_RE.match(entry['answer']):
            format_failures.append(entry)

print(f'\nNumeric format check failures: {len(format_failures)}')
for e in format_failures[:10]:
    print(f'  idx={e["original_index"]} answer={e["answer"]!r}')

# --- Self-containment check ---
BAD_PHRASES = ['from the previous', 'as above', 'part (a)', 'part (b)', 'part (c)',
               'in the previous', 'the above', 'from part']
containment_failures = []
for entry in all_entries:
    q_lower = entry['question'].lower()
    for phrase in BAD_PHRASES:
        if phrase in q_lower:
            containment_failures.append((entry, phrase))
            break

print(f'\nSelf-containment failures: {len(containment_failures)}')
for e, phrase in containment_failures[:10]:
    print(f'  idx={e["original_index"]} phrase="{phrase}" Q={e["question"][:80]}...')

# --- Deduplication (first 200 chars prefix) ---
seen_prefixes = set()
duplicates = []
deduped_entries: List[Dict] = []
for entry in all_entries:
    key = entry['question'][:200].lower().strip()
    if key in seen_prefixes:
        duplicates.append(entry)
    else:
        seen_prefixes.add(key)
        deduped_entries.append(entry)

print(f'\nDuplicates removed: {len(duplicates)}')
print(f'Final entry count after dedup: {len(deduped_entries)}')

# --- Answer type distribution ---
from collections import Counter
type_dist = Counter(e['answer_type'] for e in deduped_entries)
print(f'\nAnswer type distribution:')
for atype, count in type_dist.most_common():
    print(f'  {atype}: {count}')

# --- Flag items with notes ---
items_with_notes = [r for r in results if r.get('notes') and r['notes'].strip()]
print(f'\nItems with LLM notes: {len(items_with_notes)}')
for r in items_with_notes[:5]:
    print(f'  idx={r["original_index"]}: {r["notes"][:120]}')

Total cleaned entries: 429
Original items: 140

Numeric format check failures: 15
  idx=25 answer='(15.7, 32.6)'
  idx=39 answer='14/33'
  idx=39 answer='17/33'
  idx=45 answer='1/26'
  idx=52 answer='2630.5, 10983.0'
  idx=50 answer='31, 521'
  idx=50 answer='0.041, 0.073'
  idx=50 answer='0.04, 0.20'
  idx=51 answer='(0.233, 0.534)'
  idx=51 answer='(0.345, 0.604)'

Self-containment failures: 0

Duplicates removed: 173
Final entry count after dedup: 256

Answer type distribution:
  numeric: 215
  formula: 21
  text: 19
  boolean: 1

Items with LLM notes: 44
  idx=7: Minor discrepancy between reasoning (1130 hours) and provided answer (1132 hours), likely due to rounding differences in
  idx=5: Minor discrepancy: reasoning shows 5,323 but original answer was 5324. Used the value from detailed reasoning (5323) whi
  idx=10: The reasoning field computed n=105 and n=176, but the provided answer states n=103 and n=175. The official answers (103 
  idx=11: The answer for Part 2 uses the or

## Cell 7 — Save

In [7]:
# Use deduped entries as the final cleaned dataset
final_dataset = deduped_entries

# Ensure schema: question, reasoning, answer, answer_type, source, original_index
for entry in final_dataset:
    entry.setdefault('answer_type', 'text')
    entry.setdefault('source', 'unknown')
    entry.setdefault('original_index', -1)

save_dataset(final_dataset, config.output_file)
print(f'Saved {len(final_dataset)} entries to {config.output_file}')

# Show a few samples
print('\n--- Sample cleaned entries ---')
for entry in final_dataset[:5]:
    print(f'  [{entry["answer_type"]}] A: {entry["answer"]!r}')
    print(f'    Q: {entry["question"][:100]}...')
    print()

Saved 256 entries to ../data/master_dataset_cleaned.jsonl

--- Sample cleaned entries ---
  [numeric] A: '380'
    Q: What is the minimum sample size that will allow us to verify a 500,000-hour MTTF with 85% confidence...

  [formula] A: '$f(t) = \\frac{1}{(1+t)^2}$ for $t \\geq 0$'
    Q: Let $F(t) = 1 - (1+t)^{-1}$ for $0 \le t \le \infty$. Find the PDF $f(t)$ for this distribution....

  [numeric] A: '1'
    Q: Let $F(t) = 1 - (1+t)^{-1}$ for $0 \le t \le \infty$. Find the median lifetime $T_{50}$ for this dis...

  [text] A: 'infinity'
    Q: Let $F(t) = 1 - (1+t)^{-1}$ for $0 \le t \le \infty$. Calculate the mean of this distribution. (Hint...

  [numeric] A: '0.2841'
    Q: Three assembly plants produce the same parts. Plant A produces 25% of the volume and has a shipment ...



## Cell 8 — Statistics

In [8]:
print('=' * 80)
print('CLEANING STATISTICS')
print('=' * 80)

print(f'\nBefore: {len(master_dataset)} items')
print(f'After:  {len(final_dataset)} items')
print(f'Net change: {len(final_dataset) - len(master_dataset):+d}')

# Items that were split
split_results = [r for r in results if r.get('was_split')]
print(f'\nItems split into multiple entries: {len(split_results)}')
for r in split_results:
    orig_q = master_dataset[r['original_index']]['question'][:80]
    n_entries = len(r['entries'])
    print(f'  idx={r["original_index"]} -> {n_entries} entries: {orig_q}...')

# Classification distribution
class_dist = Counter(r.get('classification', 'UNKNOWN') for r in results if r.get('status') == 'ok')
print(f'\nClassification distribution:')
for cls, count in class_dist.most_common():
    print(f'  {cls}: {count}')

# Token usage & cost
COST_PER_1M_TOKENS = {
    'anthropic/claude-sonnet-4.5': 3.00,
}

print(f'\nToken usage & estimated cost:')
total_cost = 0
for model, tokens in token_counts.items():
    rate = COST_PER_1M_TOKENS.get(model, 1.00)
    cost = (tokens / 1_000_000) * rate
    total_cost += cost
    print(f'  {model}: {tokens:,} tokens (~${cost:.4f})')
print(f'  TOTAL: ~${total_cost:.4f}')

# Error summary
errors = [r for r in results if r.get('status') == 'error']
if errors:
    print(f'\nErrors ({len(errors)}):')
    for r in errors:
        print(f'  idx={r["original_index"]}: {r.get("error", "unknown")}')
else:
    print(f'\nNo errors!')

CLEANING STATISTICS

Before: 140 items
After:  256 items
Net change: +116

Items split into multiple entries: 113
  idx=3 -> 3 entries: Let $F(t) = 1 - (1+t)^{-1}$ , $0 \le t \le \infty$ . This is a legitimate CDF th...
  idx=4 -> 3 entries: Three assembly plants produce the same parts. Plant A produces 25% of the volume...
  idx=2 -> 3 entries: If $F(t) = 1 - e^{-\lambda t}$ (the exponential distribution), show that the MTT...
  idx=6 -> 3 entries: When the soil in an agricultural study reaches a specified level of dryness, an ...
  idx=5 -> 3 entries: Suppose we want to be 90% confident of meeting an MTTF of at least 2,000,000 hou...
  idx=1 -> 5 entries: For the distribution function $F(t) = 1 - \left(\frac{b}{b+t}\right)^a$, $0 \le ...
  idx=13 -> 3 entries: 50 devices are placed on stress for 168 hours. The probability of a device faili...
  idx=10 -> 2 entries: Using the Goal Seek function, find the necessary sample size for an LTPD = 2.21%...
  idx=11 -> 2 entries: Consider the 